<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
Supplementary code for the <a href="http://mng.bz/orYv">Build a Large Language Model From Scratch</a> book by <a href="https://sebastianraschka.com">Sebastian Raschka</a><br>
<br>Code repository: <a href="https://github.com/rasbt/LLMs-from-scratch">https://github.com/rasbt/LLMs-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="http://mng.bz/orYv"><img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>


# Multi-head Attention Plus Data Loading

In [1]:
# NBVAL_IGNORE_OUTPUT
# 打印当前 PyTorch 版本；不同版本的底层实现可能影响输出格式或性能。
from importlib.metadata import version

print("torch version:", version("torch"))

torch version: 2.12.0


The complete chapter code is located in [ch03.ipynb](./ch03.ipynb).

This notebook contains the main takeaway, multihead-attention implementation (plus the data loading pipeline from chapter 2)

## Data Loader from Chapter 2

In [2]:
import tiktoken
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader


class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        # 先把整段文本转成 token id；模型实际处理的是数字 id，不是原始字符串。
        token_ids = tokenizer.encode(txt, allowed_special={'<|endoftext|>'})

        # 用滑动窗口切出训练样本：input 是当前位置到后面 max_length 个 token。
        # target 比 input 右移一位，用来训练模型预测下一个 token。
        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i + 1: i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        # DataLoader 会调用它来知道数据集中共有多少个样本。
        return len(self.input_ids)

    def __getitem__(self, idx):
        # 返回一组 (输入序列, 目标序列)。
        return self.input_ids[idx], self.target_ids[idx]


def create_dataloader(txt, batch_size=4, max_length=256, stride=128, shuffle=True):
    # GPT-2 tokenizer 和本章后续模型设置保持一致。
    tokenizer = tiktoken.get_encoding("gpt2")

    # Dataset 负责切样本，DataLoader 负责按 batch 取样本。
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)

    # shuffle=True 会打乱样本顺序，让训练时每个 batch 更随机。
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=shuffle)

    return dataloader


# 读取一个很小的示例文本，方便快速演示数据流和注意力形状。
with open("small-text-sample.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

# encoded_text 展示文本被 tokenizer 编码后的 token id 序列。
tokenizer = tiktoken.get_encoding("gpt2")
encoded_text = tokenizer.encode(raw_text)

# vocab_size 是 GPT-2 词表大小；output_dim 是 token/position embedding 的维度。
vocab_size = 50257
output_dim = 256
max_len = 1024
context_length = max_len


# token_embedding_layer 根据 token id 查词向量。
# pos_embedding_layer 根据位置编号查位置向量，让模型知道 token 的顺序。
token_embedding_layer = nn.Embedding(vocab_size, output_dim)
pos_embedding_layer = torch.nn.Embedding(context_length, output_dim)

# 为了演示，把每条序列长度设得很短。
max_length = 4
dataloader = create_dataloader(raw_text, batch_size=8, max_length=max_length, stride=max_length)

In [3]:
for batch in dataloader:
    # x 是输入 token id，y 是右移一位后的目标 token id。
    x, y = batch

    # token embedding 表示“这个 token 是什么”。
    token_embeddings = token_embedding_layer(x)
    # position embedding 表示“这个 token 在序列中的第几个位置”。
    pos_embeddings = pos_embedding_layer(torch.arange(max_length))

    # 两者相加后，模型同时获得 token 内容和位置信息。
    input_embeddings = token_embeddings + pos_embeddings

    # 这里只取第一个 batch 做演示，所以立刻跳出循环。
    break

In [4]:
# 形状应为 (batch_size, 序列长度, embedding 维度)。
print(input_embeddings.shape)

torch.Size([8, 4, 256])


# Multi-head Attention from Chapter 3

## Variant A: Simple implementation

In [5]:
class CausalSelfAttention(nn.Module):

    def __init__(self, d_in, d_out, context_length, dropout, qkv_bias=False):
        super().__init__()
        self.d_out = d_out
        # 三个线性层分别生成 query、key、value。
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        # Dropout 用在注意力权重上，训练时随机丢弃部分连接。
        self.dropout = nn.Dropout(dropout) # New
        # 上三角 mask 用来挡住未来 token，保证自回归模型不能“偷看答案”。
        self.register_buffer('mask', torch.triu(torch.ones(context_length, context_length), diagonal=1)) # New

    def forward(self, x):
        # x 的形状是 (batch_size, token 数量, 输入维度)。
        b, n_tokens, d_in = x.shape # New batch dimension b
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)

        # 对每个 batch，query 与 key 做点积，得到 token 两两之间的相关性分数。
        attn_scores = queries @ keys.transpose(1, 2) # Changed transpose
        # 把未来位置填成 -inf，softmax 后这些位置的权重会变成 0。
        attn_scores.masked_fill_(  # New, _ ops are in-place
            self.mask.bool()[:n_tokens, :n_tokens], -torch.inf)
        # 缩放后再 softmax，可以让权重分布更稳定。
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights) # New

        # 用注意力权重加权 value，得到每个 token 的上下文表示。
        context_vec = attn_weights @ values
        return context_vec


class MultiHeadAttentionWrapper(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        # 这个包装版本直接创建多个独立的单头注意力模块。
        self.heads = nn.ModuleList(
            [CausalSelfAttention(d_in, d_out, context_length, dropout, qkv_bias)
             for _ in range(num_heads)]
        )
        # 多个头拼接后，再用线性层混合各个头的信息。
        self.out_proj = nn.Linear(d_out*num_heads, d_out*num_heads)

    def forward(self, x):
        # 每个头各自处理同一批输入，最后在最后一个维度上拼接。
        context_vec = torch.cat([head(x) for head in self.heads], dim=-1)
        return self.out_proj(context_vec)

In [6]:
torch.manual_seed(123)

# 当前演示序列长度就是数据加载器产生的 max_length。
context_length = max_length
d_in = output_dim

# 两个头各输出一半维度，拼接后回到原来的 d_in 维度。
num_heads=2
d_out = d_in // num_heads

mha = MultiHeadAttentionWrapper(d_in, d_out, context_length, 0.0, num_heads)

batch = input_embeddings
context_vecs = mha(batch)

# 输出形状仍是 (batch_size, token 数量, embedding 维度)。
print("context_vecs.shape:", context_vecs.shape)

context_vecs.shape: torch.Size([8, 4, 256])


## Variant B: Alternative implementation

In [8]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        # d_out 必须能整除头数，这样每个头才能拿到相同的 head_dim。
        assert d_out % num_heads == 0, "d_out must be divisible by num_heads"

        self.d_out = d_out
        self.num_heads = num_heads
        # 每个头只处理输出维度的一部分；所有头拼回去后等于 d_out。
        self.head_dim = d_out // num_heads # Reduce the projection dim to match desired output dim

        # 这里一次性为所有头生成 Q/K/V，然后再 reshape 切分成多个头。
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)  # Linear layer to combine head outputs
        self.dropout = nn.Dropout(dropout)
        # mask 是固定辅助张量，不参与训练，但会随模型一起移动到对应设备。
        self.register_buffer('mask', torch.triu(torch.ones(context_length, context_length), diagonal=1))

    def forward(self, x):
        # x 的形状是 (batch_size, token 数量, 输入维度)。
        b, num_tokens, d_in = x.shape

        # 先得到合并后的 Q/K/V，形状都是 (b, num_tokens, d_out)。
        keys = self.W_key(x) # Shape: (b, num_tokens, d_out)
        queries = self.W_query(x)
        values = self.W_value(x)

        # We implicitly split the matrix by adding a `num_heads` dimension
        # Unroll last dim: (b, num_tokens, d_out) -> (b, num_tokens, num_heads, head_dim)
        # 把最后一维拆成 (num_heads, head_dim)，相当于把大向量切给多个头。
        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim)
        values = values.view(b, num_tokens, self.num_heads, self.head_dim)
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)

        # Transpose: (b, num_tokens, num_heads, head_dim) -> (b, num_heads, num_tokens, head_dim)
        # 把 num_heads 放到前面，便于每个头并行计算注意力。
        keys = keys.transpose(1, 2)
        queries = queries.transpose(1, 2)
        values = values.transpose(1, 2)

        # Compute scaled dot-product attention (aka self-attention) with a causal mask
        # 对每个头分别计算 token 两两之间的注意力分数。
        attn_scores = queries @ keys.transpose(2, 3)  # Dot product for each head

        # Original mask truncated to the number of tokens and converted to boolean
        # 只截取当前输入长度对应的 mask 区域。
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]

        # Use the mask to fill attention scores
        # 被 mask 的未来位置设为 -inf，softmax 后权重为 0。
        attn_scores.masked_fill_(mask_bool, -torch.inf)

        # softmax 后，每个 token 对所有可见 token 的注意力权重和为 1。
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)

        # Shape: (b, num_tokens, num_heads, head_dim)
        # 权重乘以 value 得到各头上下文，再把 token 维放回第二维。
        context_vec = (attn_weights @ values).transpose(1, 2)

        # Combine heads, where self.d_out = self.num_heads * self.head_dim
        # contiguous() 保证内存布局连续，随后才能安全 view 回 (b, num_tokens, d_out)。
        context_vec = context_vec.contiguous().view(b, num_tokens, self.d_out)
        context_vec = self.out_proj(context_vec) # optional projection

        return context_vec

In [9]:
torch.manual_seed(123)

context_length = max_length
d_in = output_dim
# 这个高效版本内部会自己拆分多头，所以总输出维度直接设为 d_in。
d_out = d_in

mha = MultiHeadAttention(d_in, d_out, context_length, 0.0, num_heads=2)

batch = input_embeddings
context_vecs = mha(batch)

# 输出形状与输入 embedding 形状一致，表示每个 token 都得到新的上下文表示。
print("context_vecs.shape:", context_vecs.shape)

context_vecs.shape: torch.Size([8, 4, 256])
